# Generate testing-MAE data for the new PNN models (n=4, 6, 8)

`testing_hist.ipynb` (in this same folder) plots per-sample MAE histograms
from `Data/testing_data/mae_values_n{4,6,8}.npy` — those existing files come
from the **original** paper models. This notebook regenerates the same kind
of data using the **models trained this session**
(`results/model_training/PNN/n{4,6,8}/...`), so the histogram can be
re-plotted against the new models without touching the original reference
files.

**Methodology:**
1. For each model, load its own training `config.json` to get the exact
   `train_split`/`val_split`/`test_split` and `split_mode` it was trained
   with (all three used `0.8/0.1/0.1`, `"contiguous"`).
2. "Contiguous" split means no shuffling -- the test set is exactly the
   **last 10%** of that model's dataset (rows 18,000,000-20,000,000 of the
   full 20M-sample dataset, for all three models here).
3. From that test slice, draw **1000 samples at random, reproducibly** (a
   fixed-seed `numpy.random.default_rng` draw -- verified below to return
   the exact same 1000 indices every time it's run).
4. Run each trained model forward on those 1000 samples and compute
   per-sample MAE against the true (PyTheus-computed) amplitude vector --
   the same metric `ml_model.py`'s own `test_model()` uses, in the same
   precision (`float64`) each model was actually trained/evaluated in.
5. Save `mae_values_n{n}_new.npy` + `mae_summary_n{n}_new.json` under
   `Data/testing_data/` (new files, alongside the originals -- nothing
   existing is overwritten).

## Imports and configuration

In [ ]:
import sys
import json
import time
from pathlib import Path

# Add the project root to sys.path so utils.py can be imported directly.
_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if (_cwd.parent / "utils.py").exists() else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import jax.numpy as jnp
import utils

# These models were all trained with PRECISION="float64" -- match that here,
# otherwise JAX silently truncates float64 arrays to float32 on CPU/GPU.
utils.configure_precision("float64")

SAMPLE_SEED = 4        # fixed seed -> the 1000-sample draw is reproducible
N_TEST_SAMPLES = 1000

MODELS = {
    4: dict(
        model_path=REPO_ROOT / "results/model_training/PNN/n4/n4_5/best_params.msgpack",
        dataset_path=REPO_ROOT / "results/data_generation/n4/n4_0/dataset_merged.npz",
        config_path=REPO_ROOT / "results/model_training/PNN/n4/n4_5/config.json",
    ),
    6: dict(
        model_path=REPO_ROOT / "results/model_training/PNN/n6/n6_0/best_params.msgpack",
        dataset_path=REPO_ROOT / "results/data_generation/n6/n6_0/dataset_merged.npz",
        config_path=REPO_ROOT / "results/model_training/PNN/n6/n6_0/config.json",
    ),
    8: dict(
        model_path=REPO_ROOT / "results/model_training/PNN/n8/n8_1/best_params.msgpack",
        dataset_path=REPO_ROOT / "results/data_generation/n8/n8_0/dataset_merged.npz",
        config_path=REPO_ROOT / "results/model_training/PNN/n8/n8_1/config.json",
    ),
}

OUTPUT_DIR = REPO_ROOT / "plotting_code" / "Data" / "testing_data"
print("Project root:", REPO_ROOT)
print("Output dir  :", OUTPUT_DIR)

Project root: /home/bo48god/projects/deep_dreaming/PolySurrogate
Output dir  : /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Data/testing_data


## Helper functions

`get_test_slice_bounds` reads a model's own `config.json` to compute exactly
where its test split starts (no hardcoded split fractions -- if a model were
ever trained with different ratios, this would still pick the right range).

`draw_reproducible_indices` draws `N_TEST_SAMPLES` positions from within the
test slice using a fresh `np.random.default_rng(seed)` each call. Since a
freshly-seeded generator's output depends only on the seed (not on prior
random state, call order, or anything else), calling this twice with the
same seed is guaranteed to return identical indices -- verified explicitly
below, not just assumed.

In [ ]:
def get_test_slice_bounds(config_path, n_total):
    with open(config_path) as f:
        cfg = json.load(f)
    assert cfg["split_mode"] == "contiguous", "This notebook assumes a contiguous (unshuffled) split"
    n_train = int(n_total * cfg["train_split"])
    n_val = int(n_total * cfg["val_split"])
    return n_train + n_val, n_total, cfg


def draw_reproducible_indices(test_start, test_end, n_samples, seed):
    rng = np.random.default_rng(seed)
    positions = rng.choice(test_end - test_start, size=n_samples, replace=False)
    return test_start + np.sort(positions)

## Generate MAE data for each model

For each of n=4, 6, 8: locate the test slice, draw the reproducible 1000
samples (with an explicit double-draw check printed below), run the model,
compute per-sample MAE, and save the two output files.

In [ ]:
for n, info in MODELS.items():
    print(f"\n{'='*70}\nn = {n}\n{'='*70}")

    with np.load(info["dataset_path"], mmap_mode="r") as data:
        n_total = data["weights"].shape[0]
        num_edges = data["weights"].shape[1]
        n_kets = data["amps"].shape[1]

        test_start, test_end, cfg = get_test_slice_bounds(info["config_path"], n_total)
        print(f"n_total={n_total:,}  test range=[{test_start:,}:{test_end:,}) ({test_end-test_start:,} rows)")

        # Reproducibility check: draw twice with the same seed, compare.
        idx_a = draw_reproducible_indices(test_start, test_end, N_TEST_SAMPLES, SAMPLE_SEED)
        idx_b = draw_reproducible_indices(test_start, test_end, N_TEST_SAMPLES, SAMPLE_SEED)
        assert np.array_equal(idx_a, idx_b), "Sample draw is NOT reproducible!"
        print(f"Reproducibility check (draw twice, same seed={SAMPLE_SEED}): identical=True")

        t0 = time.perf_counter()
        X = np.asarray(data["weights"][idx_a], dtype=np.float64)
        Y = np.asarray(data["amps"][idx_a], dtype=np.float64)
        print(f"Loaded {len(idx_a)} samples in {time.perf_counter()-t0:.1f}s -> X{X.shape} Y{Y.shape}")

    apply_fn, params = utils.load_trained_model(
        model_type="PNN", architecture=cfg["hidden_dim"], input_dim=num_edges,
        out_dim=n_kets, nodes=n, model_path=str(info["model_path"]),
    )

    pred = apply_fn(params, jnp.asarray(X, dtype=jnp.float64))
    pred = np.asarray(pred, dtype=np.float64)

    mae = np.mean(np.abs(pred - Y), axis=-1)
    print(f"MAE: n_samples={len(mae)} mean={mae.mean():.6e} min={mae.min():.6e} max={mae.max():.6e}")

    # --- save ---
    mae_path = OUTPUT_DIR / f"mae_values_n{n}_new.npy"
    np.save(mae_path, mae.astype(np.float32))

    summary = {
        "node": n,
        "model_type": "PNN",
        "samples": int(len(mae)),
        "sample_seed": SAMPLE_SEED,
        "test_range": [int(test_start), int(test_end)],
        "dataset_path": str(info["dataset_path"]),
        "hidden_dim": cfg["hidden_dim"],
        "model_path": str(info["model_path"]),
        "input_dim": num_edges,
        "out_dim": n_kets,
        "training_seed": cfg["seed"],
        "training_precision": cfg["precision"],
        "normalize_model_output": cfg["normalize_model_output"],
        "mae_mean": float(mae.mean()),
        "mae_std": float(mae.std()),
        "mae_median": float(np.median(mae)),
        "mae_min": float(mae.min()),
        "mae_max": float(mae.max()),
        "mae_values_path": str(mae_path),
    }
    summary_path = OUTPUT_DIR / f"mae_summary_n{n}_new.json"
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(f"Saved: {mae_path}")
    print(f"Saved: {summary_path}")

print("\nAll done.")


n = 4
n_total=20,000,000  test range=[18,000,000:20,000,000) (2,000,000 rows)
Reproducibility check (draw twice, same seed=4): identical=True
Loaded 1000 samples in 7.5s -> X(1000, 24) Y(1000, 16)
MAE: n_samples=1000 mean=9.228192e-07 min=3.936111e-07 max=1.829574e-06
Saved: /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Data/testing_data/mae_values_n4_new.npy
Saved: /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Data/testing_data/mae_summary_n4_new.json

n = 6
n_total=20,000,000  test range=[18,000,000:20,000,000) (2,000,000 rows)
Reproducibility check (draw twice, same seed=4): identical=True
Loaded 1000 samples in 22.0s -> X(1000, 60) Y(1000, 64)
MAE: n_samples=1000 mean=3.913786e-06 min=1.897697e-06 max=7.530200e-06
Saved: /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Data/testing_data/mae_values_n6_new.npy
Saved: /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Data/testing_data/mae_summary_n6_new.json

n = 8


## Verify the saved files

Read back what was just written, as a final sanity check before pointing
`testing_hist.ipynb` at these files.

In [ ]:
for n in (4, 6, 8):
    arr = np.load(OUTPUT_DIR / f"mae_values_n{n}_new.npy")
    with open(OUTPUT_DIR / f"mae_summary_n{n}_new.json") as f:
        s = json.load(f)
    print(f"n={n}: shape={arr.shape} dtype={arr.dtype}  mae_mean={s['mae_mean']:.6e}  model={Path(s['model_path']).parent.name}")

n=4: shape=(1000,) dtype=float32  mae_mean=9.228192e-07  model=n4_5
n=6: shape=(1000,) dtype=float32  mae_mean=3.913786e-06  model=n6_0
n=8: shape=(1000,) dtype=float32  mae_mean=5.967337e-05  model=n8_1


---

### Next step

To plot these with `testing_hist.ipynb`, point its `MAE_FILES` dict at the
`_new` files generated here (e.g. `mae_values_n4_new.npy` instead of
`mae_values_n4.npy`) rather than editing this notebook further.

---

# Generate Jacobian sensitivity data for the new PNN models (n=4, 6, 8)

`jacobian.ipynb` (in this same folder) plots 3-panel heatmaps comparing the
**NN surrogate's Jacobian** against the **exact PyTheus Jacobian**, per node
count, from `Data/jacobian_data/{4,6,8}_node/{mean_nn,mean_pytheus,
difference}_sensitivity.npy`. Those existing files come from the
**original** paper models. This section regenerates the same kind of data
using the models trained this session, saved as new files alongside the
originals.

**What "sensitivity" is** (reverse-engineered from the original
`jacobian_summary.json`, since the plotting notebook itself only loads and
plots the matrices): it is the **Jacobian matrix**
`∂(output amplitude_i)/∂(input graph-weight_j)` — a genuine derivative,
computed two ways at the same input points:

1. Through the trained **NN surrogate** (`jax.jacrev` of the model).
2. Through the **exact PyTheus** amplitude formula
   (`utils.compute_amplitudes_no_norms` — the *unnormalised* version,
   matching `NORMED_DATA=False` / `NORMALIZE_MODEL_OUTPUT=False`, these
   models' actual training convention).

Both Jacobians are computed at the same `NUM_JACOBIAN_SAMPLES` points (drawn
reproducibly from each model's own test slice, exactly like the MAE section
above), the **absolute value** is taken of every entry, then **mean and
std across samples** give the 4 saved matrices
(`mean_nn`/`mean_py`/`std_nn`/`std_py`), plus
`difference = |mean_nn − mean_py|` and summary scalars (correlation between
the two mean matrices, mean/max absolute difference).

**A gap in the original methodology, and the choice made here:** the
original used **inconsistent sample counts** per node count (100 / 10,000 /
100) and recorded no seed at all — not reproducible as documented. Here,
`NUM_JACOBIAN_SAMPLES=1000` is used **uniformly across n=4/6/8** with an
explicit, verified seed. Note that `num_samples` only affects the
*precision* of the mean/std estimate, not the saved file size — the output
matrices are always shaped `(out_dim, input_dim)` regardless of how many
samples went into the average.

In [ ]:
import jax

JACOBIAN_DATA_DIR = REPO_ROOT / "plotting_code" / "Data" / "jacobian_data"
NUM_JACOBIAN_SAMPLES = 1000   # uniform across n=4/6/8 -- see note above

print("JAX backend:", jax.default_backend(), " devices:", jax.devices())

JAX backend: cpu  devices: [CpuDevice(id=0)]


In [ ]:
for n, info in MODELS.items():
    print(f"\n{'='*70}\nn = {n}\n{'='*70}")

    with np.load(info["dataset_path"], mmap_mode="r") as data:
        n_total = data["weights"].shape[0]
        num_edges = data["weights"].shape[1]
        n_kets = data["amps"].shape[1]

        test_start, test_end, cfg = get_test_slice_bounds(info["config_path"], n_total)

        # Reproducibility check: draw twice with the same seed, compare.
        idx_a = draw_reproducible_indices(test_start, test_end, NUM_JACOBIAN_SAMPLES, SAMPLE_SEED)
        idx_b = draw_reproducible_indices(test_start, test_end, NUM_JACOBIAN_SAMPLES, SAMPLE_SEED)
        assert np.array_equal(idx_a, idx_b), "Sample draw is NOT reproducible!"
        print(f"Reproducibility check (draw twice, same seed={SAMPLE_SEED}): identical=True")
        print(f"Drew {len(idx_a)} samples from test range [{test_start:,}:{test_end:,})")

        t0 = time.perf_counter()
        X = np.asarray(data["weights"][idx_a], dtype=np.float64)
        print(f"Loaded X{X.shape} in {time.perf_counter()-t0:.1f}s")

    edges, kets, tensor, mask = utils.build_pytheus_catalog(n, 2)
    tensor_jax = jnp.asarray(tensor, dtype=jnp.int32)
    mask_jax = jnp.asarray(mask, dtype=jnp.float32)

    apply_fn, params = utils.load_trained_model(
        model_type="PNN", architecture=cfg["hidden_dim"], input_dim=num_edges,
        out_dim=n_kets, nodes=n, model_path=str(info["model_path"]),
    )

    def nn_fn(x):
        return apply_fn(params, x[None, :])[0]

    def py_fn(x):
        x32 = x.astype(jnp.float32)
        return utils.compute_amplitudes_no_norms(x32[None, :], tensor_jax, mask_jax)[0]

    t0 = time.perf_counter()
    nn_jac = jax.vmap(jax.jacrev(nn_fn))(jnp.asarray(X, dtype=jnp.float64))
    nn_jac = jax.block_until_ready(nn_jac)
    print(f"NN Jacobian: shape={nn_jac.shape} in {time.perf_counter()-t0:.1f}s")

    t0 = time.perf_counter()
    py_jac = jax.vmap(jax.jacrev(py_fn))(jnp.asarray(X, dtype=jnp.float64))
    py_jac = jax.block_until_ready(py_jac)
    print(f"PyTheus Jacobian: shape={py_jac.shape} in {time.perf_counter()-t0:.1f}s")

    nn_jac_abs = np.abs(np.asarray(nn_jac, dtype=np.float64))
    py_jac_abs = np.abs(np.asarray(py_jac, dtype=np.float64))

    mean_nn = nn_jac_abs.mean(axis=0)
    mean_py = py_jac_abs.mean(axis=0)
    std_nn = nn_jac_abs.std(axis=0)
    std_py = py_jac_abs.std(axis=0)
    difference = np.abs(mean_nn - mean_py)

    corr = float(np.corrcoef(mean_nn.ravel(), mean_py.ravel())[0, 1])

    print(f"mean_nn shape={mean_nn.shape}  mean_py shape={mean_py.shape}")
    print(f"correlation_nn_vs_pytheus={corr:.10f}")
    print(f"mean_absolute_sensitivity_difference={difference.mean():.6e}")
    print(f"max_absolute_sensitivity_difference={difference.max():.6e}")

    out_dir = JACOBIAN_DATA_DIR / f"{n}_node"
    out_dir.mkdir(parents=True, exist_ok=True)

    np.save(out_dir / "mean_nn_sensitivity_new.npy", mean_nn.astype(np.float32))
    np.save(out_dir / "mean_pytheus_sensitivity_new.npy", mean_py.astype(np.float32))
    np.save(out_dir / "std_nn_sensitivity_new.npy", std_nn.astype(np.float32))
    np.save(out_dir / "std_pytheus_sensitivity_new.npy", std_py.astype(np.float32))
    np.save(out_dir / "difference_sensitivity_new.npy", difference.astype(np.float32))

    summary = {
        "nphotons": n,
        "target_name": "N/A (generic random dense graph test-slice samples, not target-conditioned)",
        "model_type": "PNN",
        "model_path": str(info["model_path"]),
        "dataset_path": str(info["dataset_path"]),
        "test_range": [int(test_start), int(test_end)],
        "sample_seed": SAMPLE_SEED,
        "num_samples": NUM_JACOBIAN_SAMPLES,
        "input_dim": num_edges,
        "out_dim": n_kets,
        "jacobian_shape": [n_kets, num_edges],
        "use_absolute_jacobian": True,
        "hidden_dim": cfg["hidden_dim"],
        "training_seed": cfg["seed"],
        "training_precision": cfg["precision"],
        "correlation_nn_vs_pytheus": corr,
        "mean_absolute_sensitivity_difference": float(difference.mean()),
        "max_absolute_sensitivity_difference": float(difference.max()),
    }
    summary_path = out_dir / "jacobian_summary_new.json"
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(f"Saved 5 .npy files + {summary_path}")

print("\nAll done.")


n = 4
Reproducibility check (draw twice, same seed=4): identical=True
Drew 1000 samples from test range [18,000,000:20,000,000)
Loaded X(1000, 24) in 3.5s
NN Jacobian: shape=(1000, 16, 24) in 1.6s
PyTheus Jacobian: shape=(1000, 16, 24) in 0.7s
mean_nn shape=(16, 24)  mean_py shape=(16, 24)
correlation_nn_vs_pytheus=1.0000000000
mean_absolute_sensitivity_difference=2.398498e-07
max_absolute_sensitivity_difference=5.132432e-07
Saved 5 .npy files + /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Data/jacobian_data/4_node/jacobian_summary_new.json

n = 6
Reproducibility check (draw twice, same seed=4): identical=True
Drew 1000 samples from test range [18,000,000:20,000,000)
Loaded X(1000, 60) in 11.6s
NN Jacobian: shape=(1000, 64, 60) in 1.8s
PyTheus Jacobian: shape=(1000, 64, 60) in 1.6s
mean_nn shape=(64, 60)  mean_py shape=(64, 60)
correlation_nn_vs_pytheus=1.0000000000
mean_absolute_sensitivity_difference=8.475045e-07
max_absolute_sensitivity_difference=3.893467e-06
S

## Verify the saved Jacobian files

Read back what was just written, and confirm the correlation between NN and
exact PyTheus sensitivity is essentially perfect for all three models.

In [ ]:
for n in (4, 6, 8):
    out_dir = JACOBIAN_DATA_DIR / f"{n}_node"
    mean_nn = np.load(out_dir / "mean_nn_sensitivity_new.npy")
    with open(out_dir / "jacobian_summary_new.json") as f:
        s = json.load(f)
    print(f"n={n}: shape={mean_nn.shape}  correlation={s['correlation_nn_vs_pytheus']:.10f}  "
          f"mean_abs_diff={s['mean_absolute_sensitivity_difference']:.3e}  "
          f"max_abs_diff={s['max_absolute_sensitivity_difference']:.3e}")

n=4: shape=(16, 24)  correlation=1.0000000000  mean_abs_diff=2.398e-07  max_abs_diff=5.132e-07
n=6: shape=(64, 60)  correlation=1.0000000000  mean_abs_diff=8.475e-07  max_abs_diff=3.893e-06
n=8: shape=(256, 112)  correlation=1.0000000000  mean_abs_diff=9.918e-06  max_abs_diff=1.886e-05


---

### Next step

To plot these with `jacobian.ipynb`, point its `load_jacobian_matrices()`
at the `_new` files (e.g. `mean_nn_sensitivity_new.npy` instead of
`mean_nn_sensitivity.npy`) rather than editing this notebook further.